# Copy the GitHub archive → Google Drive

Copies everything saved in `bhootbd/fao` (`archive/Meme_Template_bd`) into **My Drive / Meme_Template_bd (Telegram archive)**, then renames the files in Drive to their descriptive names from `catalog.json`.

Click **Runtime → Run all** and allow Drive access. If it stops, run all again: files already in Drive are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/fao
!git clone --depth 1 --filter=blob:none --sparse -b claude/pensive-newton-1m0wgi https://github.com/bhootbd/fao /content/fao
!cd /content/fao && git sparse-checkout set archive/Meme_Template_bd
!du -sh /content/fao/archive/Meme_Template_bd

In [ ]:
import json, os
DEST = '/content/drive/MyDrive/Meme_Template_bd (Telegram archive)'
SRC = '/content/fao/archive/Meme_Template_bd'
media = DEST + '/media'
os.makedirs(media, exist_ok=True)
# Rename files already in Drive to their catalog names first, so the copy below doesn't duplicate them
catalog = json.load(open(SRC + '/catalog.json', encoding='utf-8'))
by_key = {}
for f in os.listdir(media):
    by_key.setdefault(f.split('__')[0].rsplit('.', 1)[0], []).append(f)
renamed = 0
for e in catalog:
    target = os.path.basename(e['file'])
    for f in by_key.get(e['key'], []):
        if f != target and not os.path.exists(os.path.join(media, target)):
            os.rename(os.path.join(media, f), os.path.join(media, target))
            renamed += 1
print('renamed in Drive:', renamed)

In [ ]:
!cp -rn "$SRC"/. "$DEST"/
!cp -f "$SRC"/index.json "$SRC"/catalog.json "$DEST"/
print('files in Drive:', len(os.listdir(media)), 'of', len(os.listdir(SRC + '/media')))
drive.flush_and_unmount()
print('all changes synced to Drive')